# 🌏 Savannakhet GNN 

**Workflow:**
1. ✅ Write code locally using **VS Code**
2. ✅ Upload files to **Google Drive**
3. 👉 **Mount Drive** in Colab (This notebook)
4. 👉 Run `!python train_gnn.py` via **GPU**

---
**Required Files in Google Drive:**
```
Google Drive/
└── savannakhet-ai/
    ├── train_gnn.py                  ← Training script (from VS Code)
    └── savannakhet_real_data.json    ← Real data from DB
```

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 1: Install PyTorch Geometric                       ║
# ╚══════════════════════════════════════════════════════════╝
import subprocess, sys, torch

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu121' if torch.cuda.is_available() else 'cpu'
for pkg in ['torch_scatter', 'torch_sparse', 'torch_cluster']:
    try:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', pkg, '-q',
            '-f', f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
        ], check=True)
    except:
        pass

print(f'[OK] PyTorch: {torch.__version__} | GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'     GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 2: Mount Google Drive                              ║
# ╚══════════════════════════════════════════════════════════╝
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive mounted at /content/drive')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 3: Check Files in Drive                          ║
# ╚══════════════════════════════════════════════════════════╝
import os

# ── Update this path to match your Google Drive folder ──
DRIVE_FOLDER = '/content/drive/MyDrive/savannakhet-ai'

print(f'Checking folder: {DRIVE_FOLDER}')
if os.path.exists(DRIVE_FOLDER):
    files = os.listdir(DRIVE_FOLDER)
    print(f'[OK] Found {len(files)} files:')
    for f in files:
        size = os.path.getsize(os.path.join(DRIVE_FOLDER, f))
        print(f'     {f:<40} ({size/1024:.1f} KB)')

    # Check required files
    required = ['train_gnn.py', 'savannakhet_real_data.json']
    missing  = [r for r in required if r not in files]
    if missing:
        print(f'\n[!] Missing files: {missing}')
        print('    Please upload these files to Drive first.')
    else:
        print('\n[OK] All required files found! Ready to run.')
else:
    print(f'[ERROR] Folder not found: {DRIVE_FOLDER}')
    print('  Please create the "savannakhet-ai" folder in Google Drive')
    print('  and upload train_gnn.py and savannakhet_real_data.json')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 4: Display Edge Distribution (Pie Chart)         ║
# ╚══════════════════════════════════════════════════════════╝
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

os.chdir(DRIVE_FOLDER)
data_path = 'savannakhet_real_data.json'

if os.path.exists(data_path):
    with open(data_path, 'r', encoding='utf-8') as f:
        db = json.load(f)
    
    interactions = db.get('interactions', [])
    counts = Counter(i.get('action') for i in interactions)
    total = len(interactions)
    
    if total > 0:
        v_count = counts.get('view', 0)
        l_count = counts.get('like', 0)
        r_count = counts.get('review', 0)
        sizes = [v_count, l_count, r_count]
        
        colors = ['#3B82F6', '#10B981', '#F59E0B']
        explode = (0.03, 0.06, 0.08)
        
        fig, ax = plt.subplots(figsize=(8, 6.5))
        fig.patch.set_facecolor('#FAFBFF')
        ax.set_facecolor('#FAFBFF')
        
        wedges, texts, autotexts = ax.pie(
            sizes, labels=None, autopct='%1.1f%%', colors=colors,
            explode=explode, startangle=130, pctdistance=0.78,
            wedgeprops=dict(linewidth=2, edgecolor='white', antialiased=True),
        )
        
        if len(autotexts) > 0: autotexts[0].set_fontsize(14); autotexts[0].set_fontweight('bold'); autotexts[0].set_color('white')
        for at in autotexts[1:]:
            at.set_fontsize(11); at.set_fontweight('bold'); at.set_color('#1E293B')
        
        centre = plt.Circle((0,0), 0.42, fc='#FAFBFF', linewidth=1.5, edgecolor='#E2E8F0')
        ax.add_artist(centre)
        ax.text(0, 0.07, 'Total', ha='center', va='center', fontsize=10, color='#64748B')
        ax.text(0, -0.10, f'{total:,}', ha='center', va='center', fontsize=14, fontweight='bold', color='#1E293B')
        ax.text(0, -0.28, 'interactions', ha='center', va='center', fontsize=9, color='#64748B')
        
        legend_elements = [
            mpatches.Patch(facecolor=colors[0], edgecolor='white', label=f'View   (w=1.0) :  {v_count} ({v_count/total*100:.1f}%)'),
            mpatches.Patch(facecolor=colors[1], edgecolor='white', label=f'Like   (w=2.0) :   {l_count} ({l_count/total*100:.1f}%)'),
            mpatches.Patch(facecolor=colors[2], edgecolor='white', label=f'Review (w=3.0) :   {r_count} ({r_count/total*100:.1f}%)'),
        ]
        ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.16),
                  ncol=1, frameon=True, framealpha=0.9, fontsize=10.5, edgecolor='#CBD5E1')
        ax.set_title(f'Edge Distribution (Real Data)\nSavannakhet GNN — {total} Interactions', fontsize=12, fontweight='bold', color='#1E293B', pad=16)
        ax.axis('equal')
        plt.tight_layout()
        
        pie_chart_path = 'edge_distribution_real.png'
        plt.savefig(pie_chart_path, dpi=150, bbox_inches='tight', facecolor='#FAFBFF')
        plt.show()
        print(f'[OK] Pie Chart saved as: {pie_chart_path}')
    else:
        print('[!] No interactions found in JSON.')
else:
    print('[!] JSON data file not found. Run Step 3 to verify.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 5: Run Training Script via GPU                   ║
# ╚══════════════════════════════════════════════════════════╝

# Execute train_gnn.py
!python train_gnn.py

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 6: Display Training Chart                        ║
# ╚══════════════════════════════════════════════════════════╝
from IPython.display import Image, display
import os

chart = os.path.join(DRIVE_FOLDER, 'training_results.png')
if os.path.exists(chart):
    print('[OK] Training Results Chart:')
    display(Image(chart))
else:
    print('[!] Chart not found — run STEP 5 first.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  STEP 7: Download Model Weights & Charts               ║
# ╚══════════════════════════════════════════════════════════╝
from google.colab import files
import shutil

# Copy files to /content before downloading
for fname in ['gnn_model.pt', 'training_results.png', 'edge_distribution_real.png']:
    src = os.path.join(DRIVE_FOLDER, fname)
    dst = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        files.download(dst)
        print(f'[OK] Downloading: {fname}')
    else:
        print(f'[!] Not found: {fname}')

print()
print('[OK] Done!')
print('     Move gnn_model.pt to backend/gnn_model.pt')
print('     for production use!')